# 🐍 Clase 5 · OOP II — OrderBook y PositionTracker

> Construir un libro didáctico que contiene niveles, estabilizar las métricas sin argumentos como propiedades y migrar después a la firma pública de OrderBook sin un cambio silencioso.

**Hoy construyes:** OrderBookMini + PositionTracker; migración explícita al OrderBook estable de exchange.

⏱️ 🟢 LIVE ~15 min · 🔵 REQUIRED +18 min.

### Cómo funciona este cuaderno

Cada ejercicio declara una ruta pedagógica, decidida por contenido y no por posición: **🟢 LIVE** (núcleo presencial) · **🔵 REQUIRED** (consolidación autónoma requerida y evaluable) · **🟣 OPTIONAL** (profundización no obligatoria y no evaluable). Escribe tu respuesta, ejecuta la **✅ comprobación plegada** con `Shift+Enter` y usa la pista o solución solo cuando la necesites.

### 1. OrderBook: un objeto que contiene niveles

<sub>🟢 LIVE · núcleo presencial · ~5 min</sub>

Define `OrderBookMini(bids, asks)` donde cada lado es una lista de tuplas `(price, size)`. Es una versión didáctica deliberadamente distinta de la API pública de `exchange`.

<sub>practicas: composición</sub>

In [ ]:
class OrderBookMini:
    def __init__(self, bids, asks):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert OrderBookMini.__init__.__code__.co_consts != (None,), '⏸ implementa OrderBookMini.__init__: su cuerpo sigue siendo pass'
b = OrderBookMini([(100,1)], [(101,2)])
assert b.bids == [(100,1)] and b.asks == [(101,2)]
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class OrderBookMini:
    def __init__(self, bids, asks):
        self.bids = bids
        self.asks = asks
```

</details>

### 2. best_bid / best_ask / spread / mid

<sub>🟢 LIVE · núcleo presencial · ~5 min</sub>

Reescribe `OrderBookMini` con propiedades `best_bid`, `best_ask`, `spread` y `mid`. Son métricas calculadas sin argumentos: se leen como atributos y esa parte de la API sí permanece estable.

> 💡 `self.bids = sorted(bids, key=lambda x: -x[0])`.

<sub>practicas: @property sobre el estado</sub>

In [ ]:
class OrderBookMini:
    def __init__(self, bids, asks):
        self.bids = sorted(bids, key=lambda x: -x[0])
        self.asks = sorted(asks, key=lambda x: x[0])
    @property
    def best_bid(self):
        pass
    @property
    def best_ask(self):
        pass
    @property
    def spread(self):
        pass
    @property
    def mid(self):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert OrderBookMini.best_bid.fget.__code__.co_consts != (None,), '⏸ implementa OrderBookMini.best_bid: su cuerpo sigue siendo pass'
assert OrderBookMini.best_ask.fget.__code__.co_consts != (None,), '⏸ implementa OrderBookMini.best_ask: su cuerpo sigue siendo pass'
assert OrderBookMini.spread.fget.__code__.co_consts != (None,), '⏸ implementa OrderBookMini.spread: su cuerpo sigue siendo pass'
assert OrderBookMini.mid.fget.__code__.co_consts != (None,), '⏸ implementa OrderBookMini.mid: su cuerpo sigue siendo pass'
b = OrderBookMini([(100,1),(99,1)], [(101,1),(102,1)])
assert b.best_bid==100 and b.best_ask==101
assert b.spread==1 and b.mid==100.5
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class OrderBookMini:
    def __init__(self, bids, asks):
        self.bids = sorted(bids, key=lambda x: -x[0])
        self.asks = sorted(asks, key=lambda x: x[0])
    @property
    def best_bid(self):
        return self.bids[0][0]
    @property
    def best_ask(self):
        return self.asks[0][0]
    @property
    def spread(self):
        return self.best_ask - self.best_bid
    @property
    def mid(self):
        return (self.best_bid + self.best_ask) / 2
```

</details>

### 3. imbalance() del nivel 1

<sub>🟢 LIVE · núcleo presencial · ~5 min</sub>

Añade a `OrderBookMini` el método `imbalance()` = (bid_size − ask_size)/(bid_size + ask_size) en el mejor nivel.

<sub>practicas: otro método</sub>

In [ ]:
class OrderBookMini:
    def __init__(self, bids, asks):
        self.bids = sorted(bids, key=lambda x: -x[0])
        self.asks = sorted(asks, key=lambda x: x[0])
    def imbalance(self):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert OrderBookMini.imbalance.__code__.co_consts != (None,), '⏸ implementa OrderBookMini.imbalance: su cuerpo sigue siendo pass'
b = OrderBookMini([(100,3)], [(101,1)])
assert abs(b.imbalance() - 0.5) < 1e-9
print('ok')

<details>
<summary>💭 Pista (antes de mirar la solución)</summary>

El mejor nivel es el primero tras ordenar: `self.bids[0]` y `self.asks[0]` son tuplas `(precio, tamaño)`, así que el tamaño es el índice `[1]`.

</details>

<details>
<summary>💡 Ver solución</summary>

```python
class OrderBookMini:
    def __init__(self, bids, asks):
        self.bids = sorted(bids, key=lambda x: -x[0])
        self.asks = sorted(asks, key=lambda x: x[0])
    def imbalance(self):
        bs = self.bids[0][1]; as_ = self.asks[0][1]
        return (bs - as_) / (bs + as_)
```

</details>

### 4. PositionTracker: estado interno

<sub>🔵 REQUIRED · consolidación requerida · ~6 min</sub>

Define `PositionTracker` con `_cash=0` y `_position=0` (implementación interna) y `apply_fill(fill)` que sume `fill.cash_flow()` a la caja y `fill.size` (con signo) a la posición.

> 💡 El guión bajo dice 'tócalo con métodos, no a mano'.

<sub>practicas: encapsulación + apply_fill</sub>

In [ ]:
from exchange.trades import Fill
class PositionTracker:
    def __init__(self):
        pass
    def apply_fill(self, fill):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert PositionTracker.__init__.__code__.co_consts != (None,), '⏸ implementa PositionTracker.__init__: su cuerpo sigue siendo pass'
assert PositionTracker.apply_fill.__code__.co_consts != (None,), '⏸ implementa PositionTracker.apply_fill: su cuerpo sigue siendo pass'
t = PositionTracker()
t.apply_fill(Fill(1,'BTCUSDT','buy',100,0.5))
assert abs(t._cash + 50) < 1e-9 and abs(t._position - 0.5) < 1e-9
print('ok')

<details>
<summary>💭 Pista (antes de mirar la solución)</summary>

En `__init__`, `self._cash = 0.0` y `self._position = 0.0`. En `apply_fill`, `self._cash += fill.cash_flow()`; y la posición suma `fill.size` si es compra, lo resta si es venta.

</details>

<details>
<summary>💡 Ver solución</summary>

```python
class PositionTracker:
    def __init__(self):
        self._cash = 0.0
        self._position = 0.0
    def apply_fill(self, fill):
        self._cash += fill.cash_flow()
        self._position += fill.size if fill.side=='buy' else -fill.size
```

</details>

### 5. equity a mercado

<sub>🔵 REQUIRED · consolidación requerida · ~6 min</sub>

Reescribe `PositionTracker` añadiendo `equity(mark_price)` = `_cash + _position * mark_price`.

<sub>practicas: componer el estado</sub>

In [ ]:
from exchange.trades import Fill
class PositionTracker:
    def __init__(self):
        self._cash = 0.0
        self._position = 0.0
    def apply_fill(self, fill):
        self._cash += fill.cash_flow()
        self._position += fill.size if fill.side=='buy' else -fill.size
    def equity(self, mark_price):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert PositionTracker.equity.__code__.co_consts != (None,), '⏸ implementa PositionTracker.equity: su cuerpo sigue siendo pass'
t = PositionTracker()
t.apply_fill(Fill(1,'BTCUSDT','buy',100,1))
assert abs(t.equity(110) - 10) < 1e-9, 'compra a 100, marca a 110 -> equity 10'
print('ok')

<details>
<summary>💭 Pista (antes de mirar la solución)</summary>

Copia la clase del ejercicio anterior tal cual y añade un método más: `def equity(self, mark_price): return self._cash + self._position * mark_price`.

</details>

<details>
<summary>💡 Ver solución</summary>

```python
class PositionTracker:
    def __init__(self):
        self._cash = 0.0
        self._position = 0.0
    def apply_fill(self, fill):
        self._cash += fill.cash_flow()
        self._position += fill.size if fill.side=='buy' else -fill.size
    def equity(self, mark_price):
        return self._cash + self._position * mark_price
```

</details>

### 6. Los dos objetos, juntos

<sub>🔵 REQUIRED · consolidación requerida · ~6 min</sub>

Junta las piezas: monta un `OrderBookMini`, lee su `mid`; crea un `PositionTracker`, aplícale una compra y una venta con el `Fill` estable de L4, y guarda `eq` marcado al `mid` del libro.

> 💡 El equity se marca al `book.mid` (propiedad, sin paréntesis).

<sub>practicas: composición end-to-end</sub>

In [ ]:
from exchange.trades import Fill
class OrderBookMini:
    def __init__(self, bids, asks):
        self.bids = sorted(bids, key=lambda x: -x[0]); self.asks = sorted(asks, key=lambda x: x[0])
    @property
    def best_bid(self): return self.bids[0][0]
    @property
    def best_ask(self): return self.asks[0][0]
    @property
    def mid(self): return (self.best_bid+self.best_ask)/2
class PositionTracker:
    def __init__(self):
        self._cash=0.0; self._position=0.0
    def apply_fill(self, fill):
        self._cash += fill.cash_flow(); self._position += fill.size if fill.side=='buy' else -fill.size
    def equity(self, mark):
        return self._cash + self._position*mark
book = OrderBookMini([(99990,2.0),(99980,1.0)], [(100010,1.5)])
tracker = PositionTracker()
# aplica los dos fills y guarda eq = equity al mid del libro
eq = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert eq is not None, '⏸ eq sigue en None: completa el ejercicio antes de validar'
assert abs(book.mid - 100000) < 1e-9
assert abs(eq - 10.0) < 1e-9, 'equity al mid debe ser 10'
print('ok  eq=%.1f' % eq)

<details>
<summary>💭 Pista (antes de mirar la solución)</summary>

Tres pasos: `tracker.apply_fill(Fill(1, 'BTCUSDT', 'buy', 100000, 0.5))`, lo mismo con la venta, y `eq = tracker.equity(book.mid)`. Aquí ya reutilizas la firma pública estable de L4.

</details>

<details>
<summary>💡 Ver solución</summary>

```python
book = OrderBookMini([(99990,2.0),(99980,1.0)], [(100010,1.5)])
tracker = PositionTracker()
tracker.apply_fill(Fill(1,'BTCUSDT','buy',100000,0.5))
tracker.apply_fill(Fill(2,'BTCUSDT','sell',100050,0.2))
eq = tracker.equity(book.mid)
```

</details>

## Cierre

Composición: un OrderBook contiene niveles; un PositionTracker consume Fills. Los objetos se hablan entre sí.

Cada ejercicio lleva una ruta explícita: LIVE, REQUIRED u OPTIONAL. Sigue REQUIRED para el itinerario autónomo y elige OPTIONAL solo si tienes margen.

**L6:** seguimos construyendo el sistema sobre esta pieza.

## 🚀 Llévatelo a un `.py`

Un notebook va genial para explorar, pero el código de verdad vive en archivos `.py` que se ejecutan enteros de una vez. Abre **`book_demo.py`**: es lo que acabas de construir, ordenado y de una pieza.

Ejecútalo desde una terminal:

```bash
python book_demo.py
```

…o aquí mismo, en la siguiente celda:

In [ ]:
!python book_demo.py

> Es la misma pieza que vive en el paquete `exchange/` — aquí, condensada en un archivo que puedes leer de una sentada.